# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id` fields per the Croissant schema best practices.

### Dataset Source
This notebook loads the dataset from the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs according to Croissant schema.

In [ ]:
# List RecordSets and their @id fields
record_set_objs = metadata.record_sets
if not record_set_objs:
    print('No record sets found. Trying to access programmatically...')
    # If there are no record_set objects, try by crawling the dataset or using additional mlcroissant features
else:
    print('Available Record Sets:')
    for rs in record_set_objs:
        print(f"- RecordSet name: {getattr(rs, 'name', '')}, @id: {rs.id}")
        if hasattr(rs, 'fields'):
            print('  Fields:')
            for field in rs.fields:
                print(f"    - Field name: {getattr(field, 'name', '')}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
# For this dataset, we query all record sets and show their fields and @id's

# If no record_sets are found via metadata, let's try to get all via the records() generator
all_record_set_ids = []
try:
    if not record_set_objs or len(record_set_objs) == 0:
        # mlcroissant may allow listing them via dataset.records(record_set=None)
        print("Listing available record sets from dataset.records API...")
        # This is a listing hack to show available record_sets
        # Normally mlcroissant should show these in dataset.metadata.record_sets
        # But we will continue assuming at least one is present
    else:
        all_record_set_ids = [rs.id for rs in record_set_objs]
except Exception as e:
    print(f"Could not list record sets: {e}")

### Example: Displaying Rows From a Record Set
We'll now show a few example records from the primary tabular record set. Please refer to the above listing for the specific `@id` values used.

In [ ]:
# Identify the main data RecordSet by @id (this is typically the first or only one present)
if not all_record_set_ids:
    # Fallback: try to access all records and print
    print('No RecordSet @id identified. If this cell fails, please check the Croissant schema definition.')
else:
    main_record_set_id = all_record_set_ids[0]
    print(f"Showing sample records from RecordSet @id: {main_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for further analysis. Use the record set and field `@id`s identified above.

In [ ]:
#---
# Prepare record set IDs (as per above code, or supply explicitly if known)
# For this dataset, suppose the main table's record set @id is:
RECORD_SET_IDS = all_record_set_ids if all_record_set_ids else ['http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7862866/_/62935be8-24c5-4111-9be6-0b2e3d9593bd']
dataframes = {}

for record_set in RECORD_SET_IDS:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded RecordSet: {record_set}, shape: {df.shape}")
    else:
        print(f"No records found for RecordSet {record_set}")

# Show columns of the main dataframe
main_record_set_id = RECORD_SET_IDS[0]
if main_record_set_id in dataframes:
    print(f"Columns for RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No data extracted.")

## 4. Exploratory Data Analysis (EDA)
Let's process the main dataset. We'll select numeric fields by their field `@id`, filter on values, normalize, and group by categorical ID, following Croissant schema conventions.

In [ ]:
# For this dataset, suppose a numeric field exists (e.g., identified by its @id, such as 'http://senscience.ai/age_at_second_primary' or similar)
# Let's try to find a numeric field @id in the columns
df = dataframes.get(main_record_set_id)
if df is not None:
    print('Inferring numeric field @ids...')
    import numpy as np
    numeric_field_id = None
    for col in df.columns:
        if df[col].dtype in [np.int64, np.float64, int, float]:
            numeric_field_id = col
            break
        # Also, attempt to convert columns to numeric silently to discover 
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype in [np.int64, np.float64, int, float]:
                numeric_field_id = col
                break
        except Exception:
            pass
    if not numeric_field_id:
        print("No clear numeric field found. Please inspect DataFrame columns for numeric IDs.")
    else:
        print(f"Using numeric field @id: {numeric_field_id}")
        # Select a threshold for filtering (change as appropriate for your dataset)
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() else 1)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to group by a categorical field (e.g., sex, anatomical_location, or similar; select by @id)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No main DataFrame present for EDA.")

## 5. Visualization
Visualize distributions and relationships between fields using standard plotting libraries.

> **Note:** All references to data fields use their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if there is a numeric field and dataframe
if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Visualization not possible: no numeric field detected.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. Dataset entities were referenced by their Croissant `@id`. Key data fields were loaded dynamically, and an exploratory analysis was performed, including filtering by numeric features and grouping/categorization. Further advanced analyses can be implemented by referencing Croissant schema entities, following this template.